# Shared ConvNeXt — `custom_forward` configuration sweep

Evaluate several stage-3 block sequences via `DeltaConvNext.custom_forward` on the ImageNet validation loader.

`deltifiedStage3` layout after `rewire()`:
- indices `0 .. stage3_length-1`: shared (delta) blocks
- indices `stage3_length, stage3_length+1`: tail (LayerNorm2d + stride-2 downsample) — **must** be included so stage4 gets 768 channels

In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent))

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from timm.loss import SoftTargetCrossEntropy

from data.imagenet import ImageNetDataset
from data.transforms.transforms import build_val_transforms
from models.backbones.delta_convnext import DeltaConvNext

MODEL_PATH = Path(
    "/home/jaume/Projects/ML_and_DL/DeltaNeuralODE/outputs/outputs/shared_convnextv1_imagenet/weights/last.pth"
)
BATCH_SIZE = 96
# Set to an int for a quick smoke test; None = full val set.
MAX_BATCHES = None
NUM_CLASSES = 1000
AMP = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

device=cuda


In [2]:
def available_cpus() -> int:
    slurm_cpus = os.environ.get("SLURM_CPUS_PER_TASK")
    if slurm_cpus:
        return int(slurm_cpus)
    return len(os.sched_getaffinity(0))


def load_shared_convnext(checkpoint: Path) -> DeltaConvNext:
    """Shared-only checkpoint: no delta params allocated."""
    model = DeltaConvNext(useDeltas=False)
    model.rewire()
    ckpt = torch.load(checkpoint, map_location="cpu")
    state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    epoch = ckpt.get("epoch") if isinstance(ckpt, dict) else None
    print(f"Loaded {checkpoint}" + (f" (epoch {epoch})" if epoch is not None else ""))
    return model


model = load_shared_convnext(MODEL_PATH)
model = model.to(device)
if device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
    torch.backends.cudnn.benchmark = True
model.eval()
# Print number of parameters
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")


n_blocks = model.stage3_length
tail = [n_blocks, n_blocks + 1]  # LN + downsample
print(f"stage3_length={n_blocks}, tail indices={tail}, deltifiedStage3 len={len(model.deltifiedStage3)}")

Rewired: stage3 -> 1 shared block + shared-only (no delta params) + 2 tail layers
Loaded /home/jaume/Projects/ML_and_DL/DeltaNeuralODE/outputs/outputs/shared_convnextv1_imagenet/weights/last.pth (epoch 299)
Number of parameters: 18973768
stage3_length=9, tail indices=[9, 10], deltifiedStage3 len=11


In [3]:
val_dataset = ImageNetDataset(split="validation", transforms=build_val_transforms())
num_workers = max(1, available_cpus() // 2)
print(f"Dataloader workers: {num_workers}")


def make_val_loader() -> DataLoader:
    """Fresh loader per evaluation — avoids stale worker processes on re-iteration."""
    return DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=device.type == "cuda",
    )

Dataloader workers: 10


In [4]:
import time

from models.backbones.delta_convnext import CustomForwardConfig

criterion = SoftTargetCrossEntropy()


def _to_device(batch: torch.Tensor) -> torch.Tensor:
    batch = batch.to(device, non_blocking=True)
    if device.type == "cuda":
        batch = batch.contiguous(memory_format=torch.channels_last)
    return batch


@torch.inference_mode()
def evaluate_configuration(
    configuration: CustomForwardConfig,
    *,
    max_batches: int | None = MAX_BATCHES,
    amp: bool = AMP,
) -> dict:
    """Run val loader through `custom_forward` with the given block indices."""
    block_indices = configuration["block_indices"]
    euler_step = configuration.get("euler_step", 1.0)
    method = configuration.get("method")
    total_correct = 0.0
    total_samples = 0.0
    total_loss = 0.0
    n_steps = 0

    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    loader = make_val_loader()
    pbar = tqdm(loader, desc=f"cfg={block_indices} h={euler_step} m={method or 'RK1'}", leave=False)
    try:
        for step, (batch, y_labels) in enumerate(pbar):
            if max_batches is not None and step >= max_batches:
                break
            batch = _to_device(batch)
            y_labels = y_labels.to(device, non_blocking=True)
            soft_labels = torch.nn.functional.one_hot(y_labels, num_classes=NUM_CLASSES).float()

            with torch.autocast("cuda", enabled=amp and device.type == "cuda", dtype=torch.bfloat16):
                pred = model.custom_forward(batch, configuration)
                loss = criterion(pred, soft_labels)

            bs = pred.shape[0]
            total_samples += bs
            total_correct += (pred.argmax(1) == y_labels).sum().item()
            total_loss += loss.item() * bs
            n_steps += 1
            pbar.set_postfix(acc=f"{total_correct / total_samples:.3f}", loss=f"{total_loss / total_samples:.3f}")
    finally:
        pbar.close()
        if getattr(loader, "_iterator", None) is not None:
            loader._iterator._shutdown_workers()
            loader._iterator = None

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    it_s = n_steps / elapsed if elapsed > 0 else 0.0

    top1acc = total_correct / max(total_samples, 1)
    loss_mean = total_loss / max(total_samples, 1)
    print(
        f"  time={elapsed:.2f}s  {it_s:.2f} it/s  "
        f"top1={top1acc:.4f}  loss={loss_mean:.4f}  n={int(total_samples)}"
    )

    return {
        "configuration": configuration,
        "depth": len([i for i in block_indices if i < n_blocks]),
        "euler_step": euler_step,
        "method": method,
        "top1acc": top1acc,
        "loss": loss_mean,
        "n_samples": int(total_samples),
        "time_s": elapsed,
        "it_s": it_s,
    }


## Configurations

Edit `CONFIGURATIONS` below. Each entry is `(name, CustomForwardConfig)`.

`CustomForwardConfig` keys:
- `block_indices`: stage-3 layer indices (include the tail)
- `euler_step` (optional, default `1.0`): integration step on delta blocks only
- `method` (optional, default `None` / Euler / RK1): `"RK2"` or `"RK4"` for higher-order steps

Use `(name, None)` entries (e.g. `"-----------------"`) as section separators — they print only the label.

Defaults cover:
1. **baseline** — same order as `forward` (`0..8` + tail)
2. **shared depth sweep** — apply block `0` `n` times, then the tail
3. **subsample** — every other block, then tail
4. **euler_step sweep** — baseline indices with different step sizes

In [5]:
def cfg(
    block_indices: list[int],
    euler_step: float = 1.0,
    method: str | None = None,
) -> CustomForwardConfig:
    out: CustomForwardConfig = {"block_indices": list(block_indices) + tail, "euler_step": euler_step}
    if method is not None:
        out["method"] = method
    return out


CONFIGURATIONS: list[tuple[str, CustomForwardConfig | None]] = [
    # Block depth sweep abalation
    ("-----------------", None),
    ("baseline (forward)", cfg(list(range(n_blocks)))),
    ("D=1, ES=1", cfg([0] * 1)),
    ("D=3, ES=1", cfg([0] * 3)),
    ("D=5, ES=1", cfg([0] * 5)),
    ("D=7, ES=1", cfg([0] * 7)),
    ("D=9, ES=1", cfg([0] * 9)),
    ("D=12, ES=1", cfg([0] * 12)),
    ("D=18, ES=1", cfg([0] * 18)),
    ("D=24, ES=1", cfg([0] * 24)),
    ("D=30, ES=1", cfg([0] * 30)),
    ("D=36, ES=1", cfg([0] * 36)),
    ("D=42, ES=1", cfg([0] * 42)),
    ("D=48, ES=1", cfg([0] * 48)),
    ("D=54, ES=1", cfg([0] * 54)),
    ("D=60, ES=1", cfg([0] * 60)),

    # Euler step sweep + depth sweep
    ("-----------------", None),
    ("D=1, ES=9/1", cfg([0] * 1, euler_step=9/1)),
    ("D=3, ES=9/3", cfg([0] * 3, euler_step=9/3)),
    ("D=5, ES=9/5", cfg([0] * 5, euler_step=9/5)),
    ("D=7, ES=9/7", cfg([0] * 7, euler_step=9/7)),
    ("D=9, ES=9/9", cfg([0] * 9, euler_step=9/9)),
    ("D=12, ES=9/12", cfg([0] * 12, euler_step=9/12)),
    ("D=18, ES=9/18", cfg([0] * 18, euler_step=9/18)),
    ("D=24, ES=9/24", cfg([0] * 24, euler_step=9/24)),
    ("D=30, ES=9/30", cfg([0] * 30, euler_step=9/30)),
    ("D=36, ES=9/40", cfg([0] * 36, euler_step=9/40)),
    ("D=42, ES=9/60", cfg([0] * 42, euler_step=9/60)),
    ("D=48, ES=9/100", cfg([0] * 48, euler_step=9/100)),
    ("D=48, ES=9/200", cfg([0] * 48, euler_step=9/200)),
    ("D=48, ES=9/1000", cfg([0] * 48, euler_step=9/1000)),
    ("D=48, ES=9/10000", cfg([0] * 48, euler_step=9/1000)),

    # RK2 step sweep + depth sweep
    ("-----------------", None),
    ("D=1, ES=9/1", cfg([0] * 1, euler_step=9/1, method="RK2")),
    ("D=3, ES=9/3", cfg([0] * 3, euler_step=9/3, method="RK2")),
    ("D=5, ES=9/5", cfg([0] * 5, euler_step=9/5, method="RK2")),
    ("D=7, ES=9/7", cfg([0] * 7, euler_step=9/7, method="RK2")),
    ("D=9, ES=9/9", cfg([0] * 9, euler_step=9/9, method="RK2")),
    ("D=12, ES=9/12", cfg([0] * 12, euler_step=9/12, method="RK2")),
    ("D=18, ES=9/18", cfg([0] * 18, euler_step=9/18, method="RK2")),
    ("D=24, ES=9/24", cfg([0] * 24, euler_step=9/24, method="RK2")),
    ("D=30, ES=9/30", cfg([0] * 30, euler_step=9/30, method="RK2")),
    ("D=36, ES=9/40", cfg([0] * 36, euler_step=9/40, method="RK2")),
    ("D=42, ES=9/60", cfg([0] * 42, euler_step=9/60, method="RK2")),
    ("D=48, ES=9/100", cfg([0] * 48, euler_step=9/100, method="RK2")),
    ("D=48, ES=9/200", cfg([0] * 48, euler_step=9/200, method="RK2")),

    # RK2 step sweep + depth sweep
    ("-----------------", None),
    ("D=1, ES=9/1", cfg([0] * 1, euler_step=9/1, method="RK4")),
    ("D=3, ES=9/3", cfg([0] * 3, euler_step=9/3, method="RK4")),
    ("D=5, ES=9/5", cfg([0] * 5, euler_step=9/5, method="RK4")),
    ("D=7, ES=9/7", cfg([0] * 7, euler_step=9/7, method="RK4")),
    ("D=9, ES=9/9", cfg([0] * 9, euler_step=9/9, method="RK4")),
    ("D=12, ES=9/12", cfg([0] * 12, euler_step=9/12, method="RK4")),
    ("D=18, ES=9/18", cfg([0] * 18, euler_step=9/18, method="RK4")),
    ("D=24, ES=9/24", cfg([0] * 24, euler_step=9/24, method="RK4")),
    ("D=30, ES=9/30", cfg([0] * 30, euler_step=9/30, method="RK4")),
    ("D=36, ES=9/40", cfg([0] * 36, euler_step=9/40, method="RK4")),
    ("D=42, ES=9/60", cfg([0] * 42, euler_step=9/60, method="RK4")),
    ("D=48, ES=9/100", cfg([0] * 48, euler_step=9/100, method="RK4")),
    ("D=48, ES=9/200", cfg([0] * 48, euler_step=9/200, method="RK4")),

    # Euler step sweep
    ("-----------------", None),
    ("D=9, ES=0.01", cfg(list(range(n_blocks)), euler_step=0.01)),
    ("D=9, ES=0.125", cfg(list(range(n_blocks)), euler_step=0.125)),
    ("D=9, ES=0.25", cfg(list(range(n_blocks)), euler_step=0.25)),
    ("D=9, ES=0.5", cfg(list(range(n_blocks)), euler_step=0.5)),
    ("D=9, ES=2.0", cfg(list(range(n_blocks)), euler_step=2.0)),
    ("D=9, ES=4.0", cfg(list(range(n_blocks)), euler_step=4.0)),
    ("D=9, ES=8.0", cfg(list(range(n_blocks)), euler_step=8.0)),
    ("D=9, ES=16.0", cfg(list(range(n_blocks)), euler_step=16.0)),

    # Euler step sweep
    ("-----------------", None),
    ("D=45, ES=9/5", cfg(list(range(n_blocks)), euler_step=9/5)),
    ("D=45, ES=9/11", cfg(list(range(n_blocks)), euler_step=9/11)),
    ("D=45, ES=9/22.5", cfg(list(range(n_blocks)), euler_step=9/22.5)),
    ("D=45, ES=9/45", cfg(list(range(n_blocks)), euler_step=9/45)),
    ("D=45, ES=9/90", cfg(list(range(n_blocks)), euler_step=9/90)),
    ("D=45, ES=9/180", cfg(list(range(n_blocks)), euler_step=9/180)),
    ("D=45, ES=9/360", cfg(list(range(n_blocks)), euler_step=9/360)),
    ("D=45, ES=9/720", cfg(list(range(n_blocks)), euler_step=9/720)),
    ("D=45, ES=9/1440", cfg(list(range(n_blocks)), euler_step=9/1440)),
    ("D=45, ES=9/2880", cfg(list(range(n_blocks)), euler_step=9/2880)),

    
]

for name, c in CONFIGURATIONS:
    if c is None:
        print(name)
        continue
    idxs = c["block_indices"]
    depth = len([i for i in idxs if i < n_blocks])
    method = c.get("method") or "RK1"
    print(f"{name:22s}  depth={depth:2d}  h={c.get('euler_step', 1.0)}  m={method}  cfg={idxs}")

-----------------
baseline (forward)      depth= 9  h=1.0  m=RK1  cfg=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
D=1, ES=1               depth= 1  h=1.0  m=RK1  cfg=[0, 9, 10]
D=3, ES=1               depth= 3  h=1.0  m=RK1  cfg=[0, 0, 0, 9, 10]
D=5, ES=1               depth= 5  h=1.0  m=RK1  cfg=[0, 0, 0, 0, 0, 9, 10]
D=7, ES=1               depth= 7  h=1.0  m=RK1  cfg=[0, 0, 0, 0, 0, 0, 0, 9, 10]
D=9, ES=1               depth= 9  h=1.0  m=RK1  cfg=[0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10]
D=12, ES=1              depth=12  h=1.0  m=RK1  cfg=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10]
D=18, ES=1              depth=18  h=1.0  m=RK1  cfg=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10]
D=24, ES=1              depth=24  h=1.0  m=RK1  cfg=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10]
D=30, ES=1              depth=30  h=1.0  m=RK1  cfg=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10]
D=36, ES=1              dept

In [ ]:
results = []
for name, configuration in CONFIGURATIONS:
    if configuration is None:
        print(name)
        continue
    print(name)
    row = evaluate_configuration(configuration)
    row["name"] = name
    results.append(row)


-----------------
baseline (forward)


cfg=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10] h=1.0 m=RK1:   0%|          | 0/521 [00:00<?, ?it/s]

  time=113.40s  4.59 it/s  top1=0.8033  loss=0.8962  n=50000
D=1, ES=1


cfg=[0, 9, 10] h=1.0 m=RK1:   0%|          | 0/521 [00:00<?, ?it/s]

  time=93.00s  5.60 it/s  top1=0.3360  loss=3.6345  n=50000
D=3, ES=1


cfg=[0, 0, 0, 9, 10] h=1.0 m=RK1:   0%|          | 0/521 [00:00<?, ?it/s]

  time=105.48s  4.94 it/s  top1=0.7216  loss=1.3108  n=50000
D=5, ES=1


cfg=[0, 0, 0, 0, 0, 9, 10] h=1.0 m=RK1:   0%|          | 0/521 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

df = pd.DataFrame(results)[
    ["name", "depth", "euler_step", "top1acc", "loss", "time_s", "it_s", "n_samples", "configuration"]
]
#df = df.sort_values("top1acc", ascending=False).reset_index(drop=True)
df.to_csv("results.csv", index=False)
df


In [ ]:
# Optional: single ad-hoc config without re-running the full sweep
# evaluate_configuration(cfg([0, 0, 0, 1, 1, 1, 2, 2, 2], euler_step=0.5))